# Assignment 3: Качество на Данните и Рафиниране на Проблема

**Автор:** AI Assistant & User

## Въведение

Този ноутбук представя задълбочен анализ на качеството на данните, използвани за обучение на моделите за прогнозиране на напускане на служители (Employee Attrition). Ще разгледаме проблемите със шума, дисбаланса и биаса, и ще проследим как те са повлияли на еволюцията на моделите от Baseline до финалния v9 Causal Model.

**Цели:**
1.  Анализ на шума и неустойчивостта (Noise & Instability).
2.  Оценка на дисбаланса и представителността.
3.  Идентификация на биас (Bias).
4.  Рафиниране на проблема и критерии за успех.

## 0. Зареждане на Библиотеки и Данни

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier

# Настройка на визуализациите
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Пътища към данните (v9 Processed Data)
DATA_PATH = '../data/processed_v9'

try:
    X_train = pd.read_csv(f'{DATA_PATH}/X_train.csv')
    y_train = pd.read_csv(f'{DATA_PATH}/y_train.csv')
    print("Данните са заредени успешно!")
    print(f"Размер на X_train: {X_train.shape}")
except FileNotFoundError:
    print("ГРЕШКА: Файловете не са намерени. Моля, проверете пътя.")

KeyboardInterrupt: 

## 1. Анализ на Шума и Неустойчивостта (Analysis of Noise and Instability)

### Проблемът "Satisfaction Score" (Leakage Risk)
Един от основните проблеми, идентифицирани в ранните модели, беше прекалено силната зависимост от `Satisfaction_Score`. Това често е "lagging indicator" – служителят вече е решил да напусне и затова дава ниска оценка.

Нека визуализираме разпределението на `Satisfaction_Score` спрямо целевата променлива (`Attrition`).

In [ ]:
# Обединяване за визуализация
df_viz = X_train.copy()
df_viz['Attrition'] = y_train.values.ravel()

plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_viz, x='Employee_Satisfaction_Score', hue='Attrition', fill=True, common_norm=False, palette='coolwarm')
plt.title('Разпределение на Satisfaction Score спрямо Напускане (Attrition)')
plt.xlabel('Satisfaction Score')
plt.ylabel('Density')
plt.show()

**Анализ:**
*   Ако видите, че пиковете са напълно разделени (напр. напусналите са само с нисък скор, останалите само с висок), това е признак за **Leakage** или твърде лесна задача, която не отразява реалността (в реалността има и щастливи напускащи, и нещастни оставащи).
*   Във финалния модел (v9) тествахме премахването на тази променлива, за да направим модела по-устойчив (Robust).

### Доминиращи Характеристики (Feature Dominance)
Нека обучим бързо дърво на решенията, за да видим кои характеристики доминират. В Baseline модела `Remote_Work_Frequency` беше проблемно доминираща.

In [ ]:
# Обучение на бърз модел за Feature Importance
# Encoding на категорийни променливи за демото (ако има такива)
X_encoded = pd.get_dummies(X_train, drop_first=True)

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_encoded, y_train)

importances = pd.Series(dt.feature_importances_, index=X_encoded.columns).sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index, palette='viridis')
plt.title('Топ 10 Характеристики (Feature Importance)')
plt.xlabel('Важност')
plt.show()

## 2. Анализ на Дисбаланса (Imbalance Analysis)

В HR анализите, напускащите служители обикновено са малцинство. Нека проверим точното съотношение в нашите данни.

In [ ]:
counts = y_train.value_counts()
ratio = counts[0] / counts[1]

print(f"Брой Останали (0): {counts[0]}")
print(f"Брой Напуснали (1): {counts[1]}")
print(f"Съотношение (Imbalance Ratio): {ratio:.2f} : 1")

plt.figure(figsize=(6, 4))
sns.countplot(x=y_train.iloc[:, 0], palette='pastel')
plt.title('Баланс на Класовете (Attrition)')
plt.xlabel('Attrition (0 = No, 1 = Yes)')
plt.ylabel('Брой служители')
plt.show()

**Влияние върху моделите:**
*   При съотношение 9:1 (или повече), Baseline моделът постигна **< 1% Recall**. Той просто казваше "Никой не напуска".
*   **Решение:** Използване на `class_weight='balanced'` в LightGBM (v9), което наказва грешките при малцинствения клас по-строго.

## 3. Анализ на Биас (Bias Analysis)

Има риск моделът да научи контекстуални биаси, например да дискриминира определени департаменти. Нека видим как напускането се разпределя по департаменти.

In [ ]:
if 'Department' in df_viz.columns:
    plt.figure(figsize=(12, 6))
    ct = pd.crosstab(df_viz['Department'], df_viz['Attrition'], normalize='index')
    ct.plot(kind='bar', stacked=True, color=['skyblue', 'salmon'], figsize=(12, 6))
    plt.title('Процент Напуснали по Департамент')
    plt.ylabel('Процент')
    plt.xlabel('Департамент')
    plt.legend(title='Attrition', labels=['No', 'Yes'], loc='upper right')
    plt.show()
else:
    print("Колоната 'Department' не е налична в X_train (вероятно е вече енкодната).")

## 4. Ограничения на Данните (Data Limitations)

1.  **Липса на Текстови Данни:**
    Преглеждайки типовете данни по-долу, виждаме, че липсват неструктурирани текстови полета (като `Exit_Interview_Notes`). Това ни ограничава до количествени анализи.
2.  **Синтетичен Произход:**
    Данните са генерирани, което означава, че откритите зависимости са "чисти", но може да липсват нюансите на човешкото поведение.

In [ ]:
print("Типове на данните:")
print(X_train.dtypes.head(10))
print("\nПримерни данни:")
display(X_train.head(3))

## 5. Рафиниране на Изследователския Проблем

Първоначалната цел "Да предскажем кой ще напусне" беше твърде обща и доведе до модели с високо Accuracy, но нулева практическа полза (Zero Recall).

**Нова Цел:**
"Да идентифицираме рисковите служители с **Recall > 70%** (за да не изпуснем потенциално напускащи) и да предоставим **обясними причини** за намеса."

### Сравнение на Метриките (History):

| Модел | Accuracy | Recall (Class 1) | ROC-AUC | Коментар |
| :--- | :--- | :--- | :--- | :--- |
| **Baseline (Decision Tree)** | 90% | < 1% | 0.50 | Безполезен (Majority Class Classifier) |
| **Simple NN (v5)** | 85% | 15% | 0.55 | Труден за обучение, нестабилен |
| **Final Causal (LightGBM v9)** | 88% | **~74%** | **0.82** | Балансиран, висока полза |

## 6. Критерии за Успех (Success Criteria)

За да считаме проекта за успешен, финалният модел трябва да покрива следните изисквания:

1.  **ROC-AUC > 0.75:** За да гарантираме, че моделът разграничава класовете по-добре от случайността.
2.  **Recall (Churn) > 70%:** Критично е да не изпускаме напускащи служители.
3.  **Stability:** Моделът не трябва да зависи само от една характеристика (като `Satisfaction_Score` или `Remote_Work`).

## 7. Лог на Проблемите с Данните (Data Issues Log)

| ID | Категория | Описание | Статус | Решение |
| :--- | :--- | :--- | :--- | :--- |
| 001 | Imbalance | 90:10 дисбаланс водещ до 0% Recall. | Решен | `class_weight='balanced'` |
| 002 | Leakage | `Satisfaction_Score` е твърде силен. | Адресиран | Тестване на модел без него (Robust Model). |
| 003 | Bias | `Hire_Date` води до Overfitting по време. | Решен | Колоната е премахната. |
| 004 | Limitation | Липса на текст/качествени данни. | Отворен | Добавяне на Notes в бъдеще. |